In [1]:
import torch

try:
    # Attempt to import your model if it's in the same directory for the test execution
    from world_model import RiskWorldModel
except ImportError:
    pass

def run_era_swap(dna_tensor, world_model, env_shocks, env_feature_indices):
    """
    dna_tensor: [batch, seq_len, features] representing baseline cohort
    world_model: Trained RiskWorldModel
    env_shocks: dict of {feature_name: crisis_value}
    env_feature_indices: dict mapping feature_name to its integer index in the tensor
    """
    world_model.eval()
    with torch.no_grad():
        # 1. Calculate Baseline Risk (Safely detached from GPU/Compute Graph)
        base_risk = world_model(dna_tensor).detach().cpu().numpy()
        
        # 2. Create the Counterfactual "Crisis" Cohort
        cf_tensor = dna_tensor.clone()
        
        # Inject the macro shocks into the environment columns across all time steps
        for feature_name, shock_value in env_shocks.items():
            idx = env_feature_indices[feature_name]
            cf_tensor[:, :, idx] = shock_value
            
        # 3. Calculate Shocked Risk (Safely detached)
        cf_risk = world_model(cf_tensor).detach().cpu().numpy()
        
    uplift = cf_risk.mean() - base_risk.mean()
    print(f"Era Swap Complete! Risk shifted from {base_risk.mean()*100:.2f}% to {cf_risk.mean()*100:.2f}%")
    print(f"Total Risk Uplift: +{uplift*100:.2f}%")
    
    return cf_risk

def execute_covid_2020_shock():
    """Demonstrates the COVID-2020 Supply Chain Shock Counterfactual."""
    print("\n--- Executing Counterfactual: COVID-2020 Supply Chain Shock ---")
    
    # 1. Setup mock dimensions (In reality, these come from your data pipeline)
    batch_size = 1000  # 1000 supplier orders
    seq_len = 12       # 12 months of sequence history
    num_features = 10  # 9 DNA features + 1 Environment feature (GSCPI)
    
    # Let's map GSCPI to the last feature in the tensor
    env_feature_indices = {'GSCPI_Value': 9} 
    
    # 2. Define the COVID-2020 Macro Shock
    # Normal GSCPI is ~0.0. During early COVID/2021, it spiked to +4.3 standard deviations.
    covid_shocks = {
        'GSCPI_Value': 4.30  
    }
    
    try:
        # 3. Initialize Model and Baseline Data
        world_model = RiskWorldModel(input_dim=num_features)
        
        # Simulate a baseline cohort of suppliers operating in "Normal" times (GSCPI = 0.0)
        normal_dna_tensor = torch.randn(batch_size, seq_len, num_features)
        normal_dna_tensor[:, :, env_feature_indices['GSCPI_Value']] = 0.0 
        
        # 4. Run the Era Swap
        cf_risk = run_era_swap(
            dna_tensor=normal_dna_tensor,
            world_model=world_model,
            env_shocks=covid_shocks,
            env_feature_indices=env_feature_indices
        )
    except NameError:
        print("[Warning] RiskWorldModel not imported. Make sure world_model.py is accessible to run the test execution.")

if __name__ == "__main__":
    execute_covid_2020_shock()


--- Executing Counterfactual: COVID-2020 Supply Chain Shock ---
[Warning] RiskWorldModel not imported. Make sure world_model.py is accessible to run the test execution.
